# UDA-Hub Agentic Application

This notebook demonstrates the **UDA-Hub** multi-agent decision system for CultPass support automation.

## 1. Setup and Imports

In [1]:
from dotenv import load_dotenv
from utils import chat_interface
from agentic.workflow import orchestrator, process_ticket

load_dotenv()

False

## 2. Process Support Tickets Across Categories

### 2.1 Technical Issue: Knowledge Retrieval (RAG)

In [2]:
tech_result = process_ticket(
    {
        "text": "I cannot log in and my password reset email never arrived",
        "external_user_id": "a4ab87",
        "urgency": "high",
    },
    thread_id="session-tech-demo",
)

print("Outcome:", tech_result["outcome"])
print("Route:", tech_result["route"])
print("Confidence:", tech_result["retrieval_confidence"])
print("Response:", tech_result["response"])

Outcome: resolved
Route: knowledge
Confidence: 0.63
Response: Tap Forgot Password and confirm that you are using your registered email. Check spam or retry after 10 minutes; if it still fails, support will investigate. Source: How to Handle Login Issues [kb-login].


### 2.2 Operational Issue: Billing and Subscription Action

In [3]:
billing_result = process_ticket(
    {
        "text": "Please pause my subscription",
        "external_user_id": "f556c0",
        "execute_action": True,
    },
    thread_id="session-billing-demo",
)

print("Outcome:", billing_result["outcome"])
print("Route:", billing_result["route"])
print("Tools invoked:", [t["tool"] for t in billing_result["tool_results"]])
print("Response:", billing_result["response"])

Outcome: resolved
Route: operations
Tools invoked: ['lookup_account', 'update_subscription']
Response: You can cancel or pause under My Account, then Manage Plan. Cancellation takes effect at the end of your billing cycle. Account check: subscription is active on the basic tier. Subscription action completed: status changed from active to paused. Source: How to Cancel or Pause a Subscription [kb-cancel-pause].


### 2.3 Long-Term Memory: Preference Storage and Cross-Session Recall

In [4]:
# Turn 1: Customer states preference
res_turn1 = process_ticket(
    {
        "text": "How do I manage notifications? I prefer contact by email.",
        "external_user_id": "88382b",
    },
    thread_id="session-user-turn-1",
)
print("Turn 1 Response:", res_turn1["response"])

# Turn 2: New session for the same user recalls the preference
res_turn2 = process_ticket(
    {
        "text": "How do I change app language and location?",
        "external_user_id": "88382b",
    },
    thread_id="session-user-turn-2-new",
)
print()
print("Turn 2 (New Session) Response:", res_turn2["response"])

Turn 1 Response: Open My Account, then Notifications. You can disable marketing and recommendations, while reservation and security messages remain enabled. Account check: subscription is active on the premium tier. Source: Manage Email and Push Notifications [kb-notifications].

Turn 2 (New Session) Response: Change language or home location under My Account, then Preferences. Existing reservations will not change. Account check: subscription is active on the premium tier. I also remember: Customer prefers contact by email. Source: Change App Language and Location [kb-location-language].


### 2.4 Low Confidence / Out-of-Domain: Controlled Escalation

In [5]:
escalated_result = process_ticket(
    {
        "text": "Can CultPass arrange a private astronomy lesson on Mars?",
        "external_user_id": "a4ab87",
    },
    thread_id="session-unknown-demo",
)

print("Outcome:", escalated_result["outcome"])
print("Confidence:", escalated_result["retrieval_confidence"])
print("Response:", escalated_result["response"])

Outcome: escalated
Confidence: 0.3729
Response: I’m escalating this to a human support specialist so it can be handled safely. Reason: No sufficiently relevant knowledge was found (confidence 0.37, required 0.42). The hand-off includes category=general, urgency=normal, and attempted tools=none.


### 2.5 High-Risk Detection: Immediate Escalation

In [6]:
risk_result = process_ticket(
    {
        "text": "I think my card was stolen and this is fraud!",
        "external_user_id": "a4ab87",
        "urgency": "urgent",
    },
    thread_id="session-risk-demo",
)

print("Outcome:", risk_result["outcome"])
print("Route:", risk_result["route"])
print("Risk signals:", risk_result["classification"]["risk_signals"])
print("Response:", risk_result["response"])

Outcome: escalated
Route: escalation
Risk signals: ['fraud']
Response: I’m escalating this to a human support specialist so it can be handled safely. Reason: High-risk language requires a trained human review. The hand-off includes category=billing, urgency=urgent, and attempted tools=none.


## 3. Session State History Inspection

Short-term memory preserves conversational state within a thread session.

In [7]:
# Process a second turn in the existing technical-support session.
process_ticket(
    {"text": "The reset email is still missing", "external_user_id": "a4ab87"},
    thread_id="session-tech-demo",
)

# Inspect state history
messages = list(
    orchestrator.get_state_history(
        config={"configurable": {"thread_id": "session-tech-demo"}}
    )
)[0].values["messages"]

print("Messages in session:", [f"{type(m).__name__}: {m.content}" for m in messages])

Messages in session: ['HumanMessage: I cannot log in and my password reset email never arrived', 'AIMessage: Tap Forgot Password and confirm that you are using your registered email. Check spam or retry after 10 minutes; if it still fails, support will investigate. Source: How to Handle Login Issues [kb-login].', 'HumanMessage: The reset email is still missing', 'AIMessage: Tap Forgot Password and confirm that you are using your registered email. Check spam or retry after 10 minutes; if it still fails, support will investigate. Source: How to Handle Login Issues [kb-login].']


## 4. Interactive Chat Interface

Uncomment and run below to start an interactive chat session with UDA-Hub.

In [8]:
# chat_interface(orchestrator, {"external_user_id": "a4ab87"}, "interactive-demo")